In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
import scipy.stats as stats
import warnings
warnings.filterwarnings('ignore')

# ------------------------------------------------------------------ #
# 1. Wczytanie danych
# ------------------------------------------------------------------ #
file_name = '/dmj/fizmed/jpelczar/od_martyny/minet/signal_level_long_format_results.csv' 
try:
    df = pd.read_csv(file_name)
except FileNotFoundError:
    print(f"Nie znaleziono pliku: {file_name}")
    exit()

df['institution'] = df['institution'].astype('category')

df['method'] = df['method'].str.strip()

df['approach'] = df['classifier'].apply(
    lambda x: 'signal' if x == 'signal' else 'feature'
)

# Ustawienie punktu odniesienia (Baseline). Skrypt sprawdza co jest dostępne.
available_methods = df['method'].unique().tolist()
baseline_method = 'MINet' if 'MINet' in available_methods else available_methods[0]

df['method'] = pd.Categorical(
    df['method'],
    categories=[baseline_method] + [m for m in available_methods if m != baseline_method]
)

# ------------------------------------------------------------------ #
# 2. Model bazowy LME
# ------------------------------------------------------------------ #
def fit_lme(df, outcome='mcc'):
    if df.empty: return None
    formula = f'{outcome} ~ C(method)'
    try:
        model = smf.mixedlm(formula, data=df, groups=df['institution'])
        return model.fit(reml=True, method='lbfgs')
    except Exception as e:
        print(f"Błąd LME dla {outcome}: {e}")
        return None

print(f"\n=== GLOBAL LME MODELS (Baseline: {baseline_method}) ===")
result_mcc = fit_lme(df, outcome='mcc')
if result_mcc: print("\n[MCC LME SUMMARY]\n", result_mcc.summary())

result_auc = fit_lme(df, outcome='auc')
if result_auc: print("\n[AUC LME SUMMARY]\n", result_auc.summary())

# ------------------------------------------------------------------ #
# 3. Kontrasty feature vs signal (tylko gdy są oba typy)
# ------------------------------------------------------------------ #
if len(df['approach'].unique()) > 1:
    print("\n=== Contrast: Feature vs Signal ===")
    res_approach_mcc = smf.mixedlm('mcc ~ C(approach)', data=df, groups=df['institution']).fit(reml=True)
    print(res_approach_mcc.summary())
else:
    print("\n=== Contrast: Feature vs Signal ===")
    print("Pominięto - w pliku znajduje się tylko jedno podejście:", df['approach'].iloc[0])

# ------------------------------------------------------------------ #
# 4. & 5. Kontrasty PCA i Klasyfikatorów (TYLKO dla feature-level)
# ------------------------------------------------------------------ #
df_feature = df[df['approach'] == 'feature'].copy()

if not df_feature.empty:
    df_feature['pca'] = df_feature['pca'].astype(str) # traktujemy jako string by uniknąć błędów
    print("\n=== Feature-Level Contrasts (PCA & Classifiers) ===")
    for outcome in ['mcc', 'auc']:
        try:
            res_pca = smf.mixedlm(f'{outcome} ~ C(pca) + C(classifier)', data=df_feature, groups=df_feature['institution']).fit(reml=True)
            print(f"\n[PCA vs no-PCA ({outcome})]\n", res_pca.summary())
        except Exception: pass
else:
    print("\n=== Feature-Level Contrasts ===")
    print("Pominięto - w pliku brakuje danych dla 'feature-level' (klasycznego ML).")

# ------------------------------------------------------------------ #
# 6. Kontrasy a priori (Wilcoxon parowany po szpitalach)
# ------------------------------------------------------------------ #
PAIRS_APRIORI = [
    # DL comparisons
    ('MINet', 'MINet-MTL'),
    ('MINet', 'MINet-DANN'),
    ('MINet', 'MINet-DANN-CF'),
    ('MINet-DANN', 'MINet-DANN-CF'),
    # Classical comparisons 
    ('Unharmonised_CatBoost', 'neuroCombat_CatBoost'),
    ('PCA_CovBat_CatBoost', 'MINet-DANN-CF'),
]

def paired_comparison(df, method_a, method_b, outcome='mcc'):
    # Sprawdzenie czy obie metody w ogóle istnieją w pliku
    if method_a not in available_methods or method_b not in available_methods:
        return None
        
    a = df[df['method'] == method_a].set_index('institution')[outcome]
    b = df[df['method'] == method_b].set_index('institution')[outcome]
    
    # Parowanie idealnie po szpitalach (intersection of indexes)
    common_idx = a.index.intersection(b.index)
    a = a.loc[common_idx]
    b = b.loc[common_idx]
    diff = a - b

    n = len(diff)
    if n < 3: # Wilcoxon potrzebuje przynajmniej paru próbek
        return None

    mean_d = diff.mean()
    sd_d   = diff.std(ddof=1) if n > 1 else 1e-6
    if sd_d == 0: sd_d = 1e-6

    # Hedges' g (paired)
    J = 1 - 3 / (4 * (n - 1) - 1)
    g = J * mean_d / sd_d
    se_g = np.sqrt((1/n) + (g**2 / (2*n)))
    ci_lo, ci_hi = g - 1.96 * se_g, g + 1.96 * se_g

    # Wilcoxon signed-rank (obsługa błędów jeśli wektory są identyczne)
    try:
        stat, p_wilcoxon = stats.wilcoxon(a.values, b.values, alternative='two-sided')
    except ValueError:
        stat, p_wilcoxon = np.nan, 1.0 # Błąd zazwyczaj gdy diff == 0 wszędzie

    return {
        'method_A': method_a,
        'method_B': method_b,
        'outcome': outcome,
        'N_pairs': n,
        'mean_diff': round(mean_d, 4),
        'sd_diff': round(sd_d, 4),
        'hedges_g': round(g, 3),
        'p_wilcoxon': round(p_wilcoxon, 4),
    }

rows = []
for outcome in ['mcc', 'auc']:
    for a, b in PAIRS_APRIORI:
        res = paired_comparison(df, a, b, outcome)
        if res is not None:
            rows.append(res)

print("\n=== A priori paired comparisons (Targeted pairs) ===")
if rows:
    results_apriori = pd.DataFrame(rows)
    # Korekcja FDR (Benjamini-Hochberg) osobno dla MCC i AUC
    for outcome in ['mcc', 'auc']:
        mask = results_apriori['outcome'] == outcome
        if mask.sum() > 0:
            pvals = results_apriori.loc[mask, 'p_wilcoxon'].values
            _, p_adj, _, _ = multipletests(pvals, method='fdr_bh')
            results_apriori.loc[mask, 'p_adj_BH'] = p_adj.round(4)

    # Zmiana kolejności kolumn dla czytelności
    cols = ['outcome', 'method_A', 'method_B', 'N_pairs', 'mean_diff', 'hedges_g', 'p_wilcoxon', 'p_adj_BH']
    print(results_apriori[cols].to_string(index=False))
    results_apriori.to_csv('paired_comparisons_safe.csv', index=False)
else:
    print("Brak danych do wykonania porównań a priori z zadeklarowanej listy.")

# ------------------------------------------------------------------ #
# 7. Tabela ICC (intraclass correlation)
# ------------------------------------------------------------------ #
def compute_icc(result_lme):
    if result_lme is None: return "N/A"
    try:
        var_u = result_lme.cov_re.iloc[0, 0] 
        var_e = result_lme.scale              
        icc = var_u / (var_u + var_e)
        return round(icc, 3)
    except Exception: return "N/A"

print("\n=== Intraclass Correlation (ICC) ===")
print(f"ICC (MCC): {compute_icc(result_mcc)}")
print(f"ICC (AUC): {compute_icc(result_auc)}")